In [2]:
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets
from torchvision.transforms import v2

In [3]:
train_data = datasets.PCAM(
    root='data',
    split='train',
    download=False,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

In [4]:
train_data_sub = Subset(train_data, range(6400))
len(train_data_sub)

6400

In [5]:
train_data_sub[0][0].shape, train_data_sub[0][1]

(torch.Size([3, 96, 96]), 0)

In [6]:
batch_size = 64
data_t = DataLoader(train_data_sub, batch_size=batch_size, shuffle=True)

In [7]:
for X, y in data_t:
    print(X.shape, y.shape, y.dtype)
    break

torch.Size([64, 3, 96, 96]) torch.Size([64]) torch.int64


In [11]:
len(data_t)

100

In [8]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else 'cpu'
device

'mps'

In [41]:
class My_Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer_stack = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=16, kernel_size=(5,5), stride=1, padding=0),
            nn.BatchNorm2d(num_features=16),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=(5,5), stride=1, padding=0),
            nn.BatchNorm2d(num_features=32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=(3,3), stride=1, padding=0),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(in_features=64*9*9, out_features=1024),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(in_features=1024, out_features=64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 2)
        )

    def forward(self, x):
        logits = self.layer_stack(x)
        return logits

In [42]:
model = My_Model().to(device)
print(model)

My_Model(
  (layer_stack): Sequential(
    (0): Conv2d(3, 16, kernel_size=(5, 5), stride=(1, 1))
    (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(16, 32, kernel_size=(5, 5), stride=(1, 1))
    (5): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1))
    (9): ReLU()
    (10): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (11): Flatten(start_dim=1, end_dim=-1)
    (12): Linear(in_features=5184, out_features=1024, bias=True)
    (13): ReLU()
    (14): Dropout(p=0.5, inplace=False)
    (15): Linear(in_features=1024, out_features=64, bias=True)
    (16): ReLU()
    (17): Dropout(p=0.2, inplace=False)
    (18): Linear(in_f

In [43]:
epochs = 10
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer=optimizer, max_lr=0.001, 
                                                total_steps=len(data_t)*epochs, cycle_momentum=False)

In [44]:
def train(data_t, model, loss_fn, optimizer, scheduler):
    size = len(data_t.dataset)
    print(size)
    model.train()
    for batch, (X, y) in enumerate(data_t):
        X, y = X.to(device), y.to(device)

        optimizer.zero_grad()

        pred = model(X)
        loss = loss_fn(pred, y)

        loss.backward()
        optimizer.step()
        scheduler.step()
        

        if batch%20 == 0:
            loss = loss.item()
            print(f"loss = {loss:>7f}, [{batch}/{len(data_t)}]")
    print("last lr :", scheduler.get_last_lr()[0])

In [45]:

for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(data_t, model, loss_fn, optimizer, scheduler)

Epoch 1
-------------------------------
6400
loss = 0.693102, [0/100]
loss = 0.677310, [20/100]
loss = 0.681082, [40/100]
loss = 0.694869, [60/100]
loss = 0.693872, [80/100]
last lr : 0.0002814573614486898
Epoch 2
-------------------------------
6400
loss = 0.698731, [0/100]
loss = 0.690882, [20/100]
loss = 0.709850, [40/100]
loss = 0.711778, [60/100]
loss = 0.698892, [80/100]
last lr : 0.0007629058733040791
Epoch 3
-------------------------------
6400
loss = 0.691529, [0/100]
loss = 0.686455, [20/100]
loss = 0.681139, [40/100]
loss = 0.679032, [60/100]
loss = 0.668589, [80/100]
last lr : 0.0009999949645161446
Epoch 4
-------------------------------
6400
loss = 0.680133, [0/100]
loss = 0.677363, [20/100]
loss = 0.683416, [40/100]
loss = 0.674448, [60/100]
loss = 0.681057, [80/100]
last lr : 0.000949506469528491
Epoch 5
-------------------------------
6400
loss = 0.673775, [0/100]
loss = 0.660481, [20/100]
loss = 0.671871, [40/100]
loss = 0.679472, [60/100]
loss = 0.670219, [80/100]
las

In [46]:
from sklearn.metrics import roc_auc_score

test_data = datasets.PCAM(root='data', split='test', download=False,
                          transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]))

test_subset = Subset(test_data, range(128))

test_dl = DataLoader(test_subset, batch_size=batch_size)

def test(test_dl, model, loss_fn):
    size = len(test_dl)
    row_num = len(test_dl.dataset)
    print(size)
    test_loss, correct = 0.0, 0
    probs = []
    ys = []
    model.eval()
    with torch.no_grad():
        for X, y in test_dl:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
            probs.append(torch.softmax(pred, 1)[:, 1].cpu())
            ys.append(y.cpu())

    test_loss /= size
    correct /= row_num
    auc = roc_auc_score(torch.cat(ys), torch.cat(probs))
    print(f"Test Metrics: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")
    print(auc)


In [47]:

test(test_dl, model, loss_fn)

2
Test Metrics: 
 Accuracy: 71.9%, Avg loss: 0.630752 

0.7889204545454546


In [37]:
# epochs = 5
# for t in range(epochs):
#     print(f"Epoch {t+1}\n-------------------------------")
#     train(data_t, model, loss_fn, optimizer, scheduler)
#     test(test_dl, model, loss_fn)

In [48]:
torch.save(model.state_dict(), "model2.pth")
print("Saved PyTorch Model State to model2.pth")

Saved PyTorch Model State to model2.pth


In [49]:
model = My_Model().to(device)
model.load_state_dict(torch.load("model2.pth", weights_only=True))

<All keys matched successfully>

In [50]:
test(test_dl, model, loss_fn)

2
Test Metrics: 
 Accuracy: 71.9%, Avg loss: 0.630752 

0.7889204545454546


In [51]:
# from sklearn.metrics import roc_auc_score
# # inside eval loop:
# probs.append(torch.softmax(pred, 1)[:, 1].cpu()); ys.append(y.cpu())
# # after loop:
# auc = roc_auc_score(torch.cat(ys), torch.cat(probs))